In [1]:
# Inference Pipeline: Unwetterwarnung mit Open-Meteo API

In [2]:
# Hopsworks Feature Group laden
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent
load_dotenv(project_root / ".env")

api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")
if not api_key or not project_name:
    raise ValueError("HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen gesetzt sein.")

project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)
fs = project.get_feature_store()
weather_fg = fs.get_feature_group(
    name="weather_features_batch",
    version=1,
)
if weather_fg is None:
    raise RuntimeError("Die Feature Group weather_features_batch wurde nicht gefunden.")
print(f"✅ Feature Group geladen: {weather_fg.name} (v{weather_fg.version})")

2026-09-15 12:29:03,882 INFO: Initializing external client
2026-09-15 12:29:03,884 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-15 12:29:09,994 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167
✅ Feature Group geladen: weather_features_batch (v1)


In [3]:
# Live-Daten abrufen (Forecast von Open-Meteo)
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_live_forecast(lat: float, lon: float, location_name: str) -> pd.DataFrame:
    """
    Holt aktuelle Live-Wetterdaten + 3-Tage-Forecast von Open-Meteo
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
            "pressure_msl", "surface_pressure", "cloud_cover",
            "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cape"
        ],
        "past_days": 1,       # 🕐 Für Rolling-Window-Berechnung nötig!
        "forecast_days": 3,   # 🔮 Vorhersage-Horizont
        "timezone": "UTC"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()["hourly"]
    
    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    df["location"] = location_name
    df["latitude"] = lat
    df["longitude"] = lon
    
    print(f"✅ {len(df)} Live-Datenpunkte für {location_name} geladen")
    return df

# Definierte Standorte
LOCATIONS = {
    "Munich": (48.1351, 11.5820),
    "Hamburg": (53.5511, 9.9937)
}

In [4]:
# User-Input für Ad-hoc-Abfrage
def get_user_location_input() -> dict:
    """
    Erlaubt Ad-hoc-Abfrage für benutzerdefinierten Standort
    """
    lat = float(input("📍 Breitengrad: "))
    lon = float(input("📍 Längengrad: "))
    name = input("🏙️ Ortsname: ")
    return {"lat": lat, "lon": lon, "name": name}

In [5]:
# Real-Time Modus (Online Feature Store)
from zoneinfo import ZoneInfo

def get_live_feature_vector(event_id: str) -> dict:
    """
    Holt einen einzelnen Feature-Vektor aus der Batch Feature Group
    per Primary Key (event_id).
    """
    feature_data = weather_fg.select_all().read()
    matching_rows = feature_data[feature_data["event_id"] == event_id]
    if matching_rows.empty:
        raise KeyError(f"Kein Wetter-Feature für event_id gefunden: {event_id}")
    return matching_rows.iloc[0].to_dict()

# Beispiel: aktuellster Event für München
current_hour = datetime.now(ZoneInfo("Europe/Berlin")).replace(minute=0, second=0, microsecond=0)
current_hour = current_hour.strftime("%Y%m%d%H%M")
latitude, longitude = LOCATIONS["Munich"]
event_id = f"{latitude}_{longitude}_{current_hour}"

live_vector = get_live_feature_vector(event_id)
print(f"⚡ Feature Vector: {live_vector}")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.15s) 
⚡ Feature Vector: {'location_id': '48.1351_11.582', 'location_name': 'Muenchen', 'latitude': 48.1351, 'longitude': 11.582, 'time': Timestamp('2026-09-15 12:00:00+0000', tz='Etc/UTC'), 'temperature_2m': 23.51500129699707, 'relative_humidity_2m': 50.0, 'precipitation': 0.0, 'pressure_msl': 1019.7000122070312, 'surface_pressure': 960.647705078125, 'cloud_cover': 0.0, 'wind_speed_10m': 5.052840709686279, 'wind_gusts_10m': 5.052840709686279, 'cape': 0.0, 'precip_rolling_sum_3h': 0.0, 'precip_rolling_sum_6h': 0.0, 'precip_rolling_sum_12h': 0.0, 'wind_gust_max_3h': 5.154415607452393, 'wind_gust_max_6h': 5.154415607452393, 'wind_gust_max_12h': 5.154415607452393, 'pressure_mean_3h': 1020.6666666666666, 'pressure_mean_6h': 1021.5166625976562, 'pressure_mean_12h': 1022.566660563151, 'pressure_change_3h': -2.29998779296875, 'pressure_drop_rate': -0.76666259765625, 'wind_gust_anomaly': 1.042416137953599, 'temp_cha

In [6]:
# Modell aus der Model Registry herunterladen
import joblib
import os

mr = project.get_model_registry()

# Neuestes Modell abrufen (oder spezifische Version)
model_meta = mr.get_model(
    name="severe_weather_classifier",
    version=1  # oder weglassen für automatisch neueste Version
)

# Modell-Verzeichnis herunterladen
model_dir = model_meta.download()
print(f"✅ Modell heruntergeladen nach: {model_dir}")

# Modell laden
model_path = os.path.join(model_dir, "model.joblib")
model = joblib.load(model_path)

print(f"✅ Modell geladen: {model_meta.name} (v{model_meta.version})")
print(f"📊 Trainings-Metriken: {model_meta.training_metrics}")

Downloading: 0.000%|          | 0/152275 elapsed<00:00 remaining<?

Downloading: 0.000%|          | 0/102 elapsed<00:00 remaining<?

✅ Modell heruntergeladen nach: /tmp/hopsworks/models/fhnw_p1_weather_forcasts/severe_weather_classifier/1/severe_weather_classifier_1
✅ Modell geladen: severe_weather_classifier (v1)
📊 Trainings-Metriken: {'n_test_samples': 71.0, 'n_train_samples': 281.0, 'f1_score': 0.0, 'positive_class_ratio': 0.0}


In [7]:
# Real-Time Single Prediction
def run_realtime_prediction(model, feature_vector: dict, threshold: float = 0.3) -> dict:
    """
    Führt Echtzeit-Prediction für einen einzelnen Feature-Vektor durch.
    """
    expected_features = getattr(model, "feature_names_in_", None)
    if expected_features is None:
        expected_features = model.get_booster().feature_names
    if expected_features is None or len(expected_features) == 0:
        raise ValueError("Das Modell enthält keine Feature-Namen für die Inference.")

    missing_features = [
        feature_name for feature_name in expected_features
        if feature_name not in feature_vector
    ]
    if missing_features:
        raise KeyError(f"Fehlende Modell-Features: {missing_features}")

    X = pd.DataFrame([
        {feature_name: feature_vector[feature_name] for feature_name in expected_features}
    ])
    X = X.apply(pd.to_numeric, errors="coerce")

    probability = model.predict_proba(X)[0, 1]
    warning = bool(probability >= threshold)

    result = {
        "storm_probability": round(float(probability), 4),
        "storm_warning": warning,
        "risk_level": "🔴 HOCH" if probability >= 0.6 else
                       "🟠 MITTEL" if probability >= threshold else "🟢 NIEDRIG"
    }
    return result

# Ausführen
result = run_realtime_prediction(model, live_vector, threshold=0.3)
print(f"⚡ Real-Time Ergebnis: {result}")

⚡ Real-Time Ergebnis: {'storm_probability': 0.0, 'storm_warning': False, 'risk_level': '🟢 NIEDRIG'}
